# Correlated Inputs in Uncertainty Propagation

The default `Particles(N, dist)` call draws **independent** samples for each variable.
When inputs are statistically correlated, the independence assumption is wrong and the
propagated uncertainty will be biased — typically underestimated for positively correlated inputs.

There are two real approaches for generating correlated particle inputs:

1. **Native MCM** — `Particles(N, multivariate_dist)` when a suitable multivariate distribution
   exists in Distributions.jl (multivariate Normal, LogNormal, t, …)
2. **Copulas.jl** — the general solution for arbitrary combinations of marginals
   or non-Gaussian dependence structures

Once the `Particles` objects are built, all arithmetic and statistics work identically in both cases.

In [ ]:
using MonteCarloMeasurements, Distributions, LinearAlgebra, Random, Statistics
Random.seed!(42)
N = 10_000

## Why Correlation Matters

Consider two standard-normal inputs X and Y with correlation ρ.
For the simple sum Z = X + Y, probability theory gives:

$$\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y) + 2\,\text{Cov}(X, Y) = 2 + 2\rho$$

| Correlation ρ | True Std(Z) | Naive independent |
|:---:|:---:|:---:|
| 0.0 | 1.414 | 1.414 ✓ |
| 0.5 | 1.732 | 1.414 ✗ |
| 0.8 | 1.897 | 1.414 ✗ |

Positive correlation **inflates** output uncertainty; ignoring it produces intervals
that are too narrow and give a false sense of precision.
For negative correlation the bias goes the other way.

In [ ]:
for ρ in [0.0, 0.5, 0.8]
    println("ρ = ", ρ,
            "  true Std(X+Y) = ", round(sqrt(2 + 2ρ), digits=3),
            "  naive = ", round(sqrt(2.0), digits=3))
end

## Approach 1 — Native MCM: `Particles(N, multivariate_dist)`

When the argument to `Particles` is a **multivariate** distribution,
MonteCarloMeasurements.jl returns a `Vector{Particles}` where all elements
share the same jointly-drawn sample matrix — so they are automatically correlated.

This works for any multivariate distribution implemented in Distributions.jl:

| Distribution | Marginals |
|---|---|
| `MvNormal(μ, Σ)` | all Normal |
| `MvLogNormal(MvNormal(μ, Σ))` | all LogNormal |
| `MvTDist(ν, μ, Σ)` | all t-distributed |
| `Dirichlet(α)` | all on (0,1), summing to 1 |

The helper functions `pmean(p)`, `pcov(p)` and `pcor(p)` operate on a `Vector{Particles}`
and return the sample mean vector, covariance matrix and correlation matrix; `pstd.(p)` gives the standard deviations.
On a single `Particles` value use `pmean`, `pstd` and `pquantile`: the plain `mean`, `std` and `quantile`
treat it as one number and do not summarise the particles.

In [ ]:
ρ = 0.6
Σ = [1.0 ρ; ρ 1.0]

# One call — returns a 2-element Vector{Particles{Float64,N}}
p = Particles(N, MvNormal([0.0, 0.0], Σ))
X, Y = p[1], p[2]

println("Particle covariance (pcov):")
display(round.(pcov(p), digits=4))
println()
println("Std(X+Y): ", round(pstd(X + Y), digits=4), "  (true: ", round(sqrt(2 + 2ρ), digits=4), ")")

In [ ]:
# MvLogNormal — parameters are the mean and covariance of the underlying normal (log-space)
p_ln = Particles(N, MvLogNormal(MvNormal([2, 1], [2. 1; 1 3])))
println("MvLogNormal particles: ", p_ln)
println()
println("Log-space means (should recover [2, 1]):")
display(round.(pmean(log.(p_ln)), digits=4))
println()
println("Log-space covariance (should recover [2 1; 1 3]):")
display(round.(pcov(log.(p_ln)), digits=4))

In [ ]:
# Three correlated variables — full correlation matrix
μ3 = [5.0, 10.0, 2.0]
σ  = [0.5,  1.0, 0.2]
R  = [1.0 0.4 0.2;
      0.4 1.0 0.6;
      0.2 0.6 1.0]
Σ3 = Diagonal(σ) * R * Diagonal(σ)   # covariance from correlation matrix + std devs

p3 = Particles(N, MvNormal(μ3, Σ3))  # 3-element Vector{Particles}
A, B, C = p3[1], p3[2], p3[3]

Kcov = pcov(p3)
println("Correlations (sample vs. target):")
println("  ρ(A,B) = ", round(Kcov[1,2]/(pstd(A)*pstd(B)), digits=3), "  (target 0.4)")
println("  ρ(A,C) = ", round(Kcov[1,3]/(pstd(A)*pstd(C)), digits=3), "  (target 0.2)")
println("  ρ(B,C) = ", round(Kcov[2,3]/(pstd(B)*pstd(C)), digits=3), "  (target 0.6)")

### What happens under the hood — the Cholesky connection

`MvNormal` internally draws correlated samples by applying the Cholesky factor L of Σ
(where L Lᵀ = Σ) to a vector of independent standard normals **z**:

$$\mathbf{x} = \boldsymbol{\mu} + L\,\mathbf{z}, \qquad \mathbf{z} \sim \mathcal{N}(\mathbf{0}, I)$$

You can replicate this manually, which is occasionally useful if you need to
reuse the same draws **z** across multiple scenarios:

```julia
ρ = 0.3
Σ = [1.0 ρ; ρ 1.0]
L = cholesky(Σ).L
z = randn(2, N)          # independent draws — reuse these across scenarios
xy = L * z               # correlated samples
X = Particles(xy[1, :])
Y = Particles(xy[2, :])
```

In all other situations, prefer `Particles(N, MvNormal(...))` directly.

## Approach 2 — Copulas.jl for Arbitrary Marginals

Approach 1 is limited to multivariate distributions that exist in Distributions.jl —
in practice, mainly the multivariate normal family.
When marginals are of **mixed or non-normal types** (e.g. one Exponential and one Beta),
or when you need **non-Gaussian tail dependence**, Copulas.jl is the general solution.

A copula separates the marginal distributions from the dependence structure:

$$F(x, y) = C\bigl(F_X(x),\, F_Y(y)\bigr)$$

The `SklarDist` type in Copulas.jl combines any copula with any Distributions.jl marginals.

In [ ]:
# Install if needed:  using Pkg; Pkg.add("Copulas")
using Copulas

# Gaussian copula — Normal and Exponential marginals with ρ ≈ 0.6
# (MvNormal cannot represent this combination)
C     = GaussianCopula([1.0 0.6; 0.6 1.0])
joint = SklarDist(C, (Normal(0.0, 1.0), Exponential(1.0)))

s = rand(joint, N)         # 2×N matrix
X = Particles(s[1, :])     # ~ Normal(0, 1)
Y = Particles(s[2, :])     # ~ Exponential(1)

println("Pearson correlation:  ", round(cor(s[1, :], s[2, :]), digits=4))
ranks(v) = invperm(sortperm(v))  # rank of each value (no ties for continuous draws)
println("Spearman correlation: ", round(cor(ranks(s[1, :]), ranks(s[2, :])), digits=4))
println()
result = X + Y
println("Mean of X+Y:  ", round(pmean(result), digits=4), "  (true: 1.0)")
println("Std  of X+Y:  ", round(pstd(result),  digits=4))

In [ ]:
# Clayton copula — strong lower-tail dependence; Gaussian copula has symmetric tails
# Useful when extreme low values of X and Y tend to occur together
C_clayton = ClaytonCopula(2, 3.0)
joint_c   = SklarDist(C_clayton, (LogNormal(0.0, 0.5), LogNormal(0.0, 0.3)))

sc = rand(joint_c, N)
X2 = Particles(sc[1, :])
Y2 = Particles(sc[2, :])

result2 = X2 * Y2
println("Product of two Clayton-correlated log-normals:")
println("  Mean:    ", round(pmean(result2),            digits=4))
println("  Std:     ", round(pstd(result2),             digits=4))
println("  95% CI: [", round(pquantile(result2, 0.025), digits=4),
        ", ",          round(pquantile(result2, 0.975), digits=4), "]")

## Worked Example — Uncertain Area Measurement

A rectangular plot is measured with the same surveying instrument.
Because the same instrument is used, errors in length L and width W are positively
correlated (ρ = 0.4). Both marginals are Normal, so Approach 1 applies directly.

The delta-method analytical benchmark is:

$$\text{Var}(A) \approx \bar{W}^2\,\sigma_L^2 + \bar{L}^2\,\sigma_W^2 + 2\,\bar{L}\,\bar{W}\,\text{Cov}(L,W)$$

In [ ]:
μ_L, σ_L = 10.0, 0.2   # length: 10 ± 0.2 m
μ_W, σ_W =  5.0, 0.1   # width:   5 ± 0.1 m
ρ_LW = 0.4

# Analytical benchmark (delta method)
cov_LW    = ρ_LW * σ_L * σ_W
var_A     = μ_W^2*σ_L^2 + μ_L^2*σ_W^2 + 2μ_L*μ_W*cov_LW
var_A_ind = μ_W^2*σ_L^2 + μ_L^2*σ_W^2
println("--- Analytical (delta method) ---")
println("Std(A) with correlation:    ", round(sqrt(var_A),     digits=4), " m²")
println("Std(A) without correlation: ", round(sqrt(var_A_ind), digits=4), " m²")
println()

# Approach 1 — native MCM
Σ_LW = [σ_L^2        ρ_LW*σ_L*σ_W;
        ρ_LW*σ_L*σ_W  σ_W^2      ]
p_LW   = Particles(N, MvNormal([μ_L, μ_W], Σ_LW))
A_corr = p_LW[1] * p_LW[2]

# Naive independent
A_ind = Particles(N, Normal(μ_L, σ_L)) * Particles(N, Normal(μ_W, σ_W))

println("--- Monte Carlo (N = ", N, ") ---")
println("Std(A) with correlation:    ", round(pstd(A_corr), digits=4), " m²")
println("Std(A) without correlation: ", round(pstd(A_ind),  digits=4), " m²")
println()
println("95% CI with correlation:    [",
        round(pquantile(A_corr, 0.025), digits=3), ", ",
        round(pquantile(A_corr, 0.975), digits=3), "] m²")
println("95% CI without correlation: [",
        round(pquantile(A_ind, 0.025), digits=3), ", ",
        round(pquantile(A_ind, 0.975), digits=3), "] m²")

## Summary

| Situation | Approach |
|---|---|
| All marginals are Normal | `Particles(N, MvNormal(μ, Σ))` |
| All marginals are LogNormal | `Particles(N, MvLogNormal(MvNormal(μ, Σ)))` |
| All marginals are t-distributed | `Particles(N, MvTDist(ν, μ, Σ))` |
| Mixed or non-standard marginals | Copulas.jl — `SklarDist(copula, marginals)` |
| Non-Gaussian tail dependence | Copulas.jl — Clayton, Gumbel, Frank copulas |

The key takeaway: **positive correlation inflates output uncertainty**; ignoring it
produces intervals that are too narrow and give a false sense of precision.